# Multi-Agent Deep Research System Integration

This notebook demonstrates and executes the complete multi-agent research pipeline, including:
1. **Scoping**: User clarification and research brief generation.
2. **Coordination**: Supervisor agent delegating subtopics to specialized research agents.
3. **Research**: Parallel tool-based web searching with domain filtering and reflection.
4. **Synthesis**: Compiling compressed factual findings into a high-integrity final report.

In [ ]:
import sys
import os
from dotenv import load_dotenv

# Ensure project root is in the path
project_root = os.path.abspath("..")
sys.path.append(project_root)

load_dotenv(os.path.join(project_root, ".env"))

# Verify environment keys are loaded
print("Tavily API Key exists:", bool(os.getenv("TAVILY_API_KEY")))
print("Groq API Key exists:", bool(os.getenv("GROQ_API_KEY")))

In [ ]:
import asyncio
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END

from deepresearch.utils import get_today_str
from deepresearch.prompt import final_report_generation_prompt
from deepresearch.state import AgentState, AgentInputState
from deepresearch.agents.scoping_agent import clarify_with_user, write_research_brief
from deepresearch.agents.supervisor_agent import supervisor_agent

# Ensure nested asyncio is applied for Jupyter notebook cell executions
try:
    import nest_asyncio
    nest_asyncio.apply()
    print("nest_asyncio applied successfully.")
except ImportError:
    print("nest_asyncio not available.")

In [ ]:
_DEFAULT_FINAL_REPORT_MODEL = "groq:qwen/qwen3-32b"

async def final_report_generation(state: AgentState, config: RunnableConfig):
    """
    Final report generation node.
    
    Synthesizes all collected research findings into a comprehensive final report.
    """
    cfg = config.get("configurable", {})
    model_name = cfg.get("final_report_model", _DEFAULT_FINAL_REPORT_MODEL)
    writer_model = init_chat_model(model_name, temperature=0.3) # Low temperature to prevent fact drift

    notes = state.get("notes", [])
    findings = "\n\n".join(notes)

    final_report_prompt = final_report_generation_prompt.format(
        research_brief=state.get("research_brief", ""),
        findings=findings,
        date=get_today_str()
    )

    final_report = await writer_model.ainvoke([HumanMessage(content=final_report_prompt)])
    
    return {
        "final_report": final_report.content,
        "messages": ["Here is the final report: " + final_report.content],
    }

In [ ]:
# Construct the workflow
deep_researcher_builder = StateGraph(AgentState, input_schema=AgentInputState)

# Add workflow nodes
deep_researcher_builder.add_node("clarify_with_user", clarify_with_user)
deep_researcher_builder.add_node("write_research_brief", write_research_brief)
deep_researcher_builder.add_node("supervisor_subgraph", supervisor_agent)
deep_researcher_builder.add_node("final_report_generation", final_report_generation)

# Add workflow edges
deep_researcher_builder.add_edge(START, "clarify_with_user")
deep_researcher_builder.add_edge("write_research_brief", "supervisor_subgraph")
deep_researcher_builder.add_edge("supervisor_subgraph", "final_report_generation")
deep_researcher_builder.add_edge("final_report_generation", END)

# Compile the full workflow
agent = deep_researcher_builder.compile()
print("Workflow compiled successfully.")

In [ ]:
async def run_research():
    # Configure settings: allow_clarification=False to bypass scoping conversation and start research directly
    config = {
        "configurable": {
            "thread_id": "test_run_1",
            "allow_clarification": False,
            "max_react_tool_calls": 3,
            "min_search_iterations": 2,
            "max_supervisor_history": 4
        }
    }
    
    query = "Compare the number of IPL titles won by Chennai Super Kings versus Mumbai Indians."
    print(f"Starting research for: '{query}'\n")
    
    initial_state = {
        "messages": [HumanMessage(content=query)]
    }
    
    result = await agent.ainvoke(initial_state, config=config)
    
    print("\n================ FINAL REPORT ================\n")
    print(result.get("final_report", "No report generated."))

# Execute the async runner
await run_research()